In [3]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
vs_index = VectorSearchIndex(
    keyword_fields = ['course'],
    mode = 'ivf',
    db_path = 'faq_vectors2.db'
)

In [16]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)

In [21]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results = 5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}
        
        return self.index.search(
            query_vector,
            num_results = num_results,
            filter_dict = filter_dict,
        )
vector_assistant = RAGVector(
    embedder = model,
    index = vs_index,
    llm_client = openai_client,
)

In [20]:
vector_assistant.rag("the program has already begun, can I still sign up?")


'You didn\'t provide a context. Please provide the context, and I\'ll be happy to help answer your question. If no context is provided, my answer would be "I don\'t know."'

In [22]:
vs_index.close()